# Lesson 4: Logging & Configuration

**Week 4 · Data Engineering Course**

---

When a pipeline runs overnight and fails at 3 AM, `print()` statements are useless — you need a **log file**. When you deploy a pipeline to a server, the database password cannot be hard-coded in the notebook — it needs to come from a **config file** or **environment variable**.

This lesson covers:
- Python's `logging` module — structured, timestamped messages that go to a file
- Configuration files — storing settings in JSON so you can change them without touching the code
- Environment variables — the standard way to pass secrets to a pipeline
- `.env` files — a convenient way to set environment variables locally
- Putting it all together in a configurable, logged pipeline

In [ ]:
import logging
import json
import os
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime

DATA  = Path('data')
LOGS  = Path('logs')
LOGS.mkdir(exist_ok=True)

print('Ready.')

---

## 4.1 Why Not Just Use print()?

`print()` is fine during development, but it has problems in production:

| `print()` | `logging` |
|-----------|----------|
| No timestamp — you cannot tell when something happened | Every message has a precise timestamp |
| No severity — everything looks the same | Messages have levels: DEBUG, INFO, WARNING, ERROR |
| Only goes to the screen | Can write to a file, a screen, and a monitoring service at the same time |
| You have to delete it or comment it out for production | Just change the log level to hide debug messages in production |

---

## 4.2 Basic Logging

In [ ]:
# Configure logging — do this once at the top of your script
logging.basicConfig(
    level=logging.INFO,           # show INFO and above (INFO, WARNING, ERROR, CRITICAL)
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)

# The five logging levels (lowest to highest)
logging.debug('This is debug — detailed info for diagnosing problems')   # not shown at INFO level
logging.info('This is info — normal pipeline progress messages')
logging.warning('This is a warning — something unexpected, but not fatal')
logging.error('This is an error — something failed')
logging.critical('This is critical — the whole pipeline must stop')

In [ ]:
# Use f-strings to add context to log messages
city = 'Lagos'
rows = 150
logging.info(f'Fetched {rows} rows for {city}')

try:
    result = 1 / 0
except ZeroDivisionError as e:
    logging.error(f'Failed to process {city}: {e}')

---

## 4.3 Logging to a File

In production, you want logs saved to a file so you can read them later. You can log to both the screen and a file at the same time using **handlers**.

In [ ]:
# Build a logger that writes to BOTH the console and a file
def build_logger(name, log_dir=LOGS):
    '''Create a named logger that writes to console and a dated log file.'''
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)   # capture everything; handlers control what they show

    formatter = logging.Formatter(
        fmt='%(asctime)s  %(levelname)-8s  %(name)s  %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S',
    )

    # Console handler — show INFO and above
    console = logging.StreamHandler()
    console.setLevel(logging.INFO)
    console.setFormatter(formatter)

    # File handler — save DEBUG and above to a file
    today = datetime.now().strftime('%Y-%m-%d')
    log_file = log_dir / f'{name}_{today}.log'
    file_handler = logging.FileHandler(log_file, encoding='utf-8')
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    logger.addHandler(console)
    logger.addHandler(file_handler)
    return logger, log_file

pipeline_log, log_path = build_logger('weather_pipeline')
pipeline_log.info('Pipeline started')
pipeline_log.debug('This debug message goes to the file but not the console')
pipeline_log.warning('This warning goes to both')

print(f'\nLog file: {log_path}')

In [ ]:
# Read the log file to confirm the debug message is there
if log_path.exists():
    print(log_path.read_text(encoding='utf-8'))

---

## 4.4 Configuration Files

Hard-coding values in a script is bad practice:
- You have to edit the code to change a URL or a timeout
- Different environments (local, staging, production) need different settings
- You might accidentally commit a secret

The solution is a **config file** — a JSON file that holds all the settings.

In [ ]:
# Create a config.json for the weather pipeline
config = {
    'api': {
        'base_url':    'https://api.open-meteo.com/v1/forecast',
        'timeout_sec': 10,
        'fields':      'temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max',
        'sleep_between_requests': 0.3
    },
    'cities_file':   'data/cities.csv',
    'output_dir':    'data/clean',
    'log_dir':       'logs',
    'db': {
        'host':   'localhost',
        'port':   5432,
        'dbname': 'weather_db',
        'user':   'postgres'
    }
}

# Note: password is NOT in the config file — it will come from an environment variable

config_path = Path('config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

print('Created config.json:')
print(config_path.read_text(encoding='utf-8'))

In [ ]:
# Load and use the config in your script
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

print('API base URL:', cfg['api']['base_url'])
print('Timeout:',      cfg['api']['timeout_sec'], 'seconds')
print('DB host:',      cfg['db']['host'])

---

## 4.5 Environment Variables

An **environment variable** is a value stored in the operating system, outside your code. It is the standard way to pass secrets (passwords, API keys) to a program.

Setting one in the terminal:
```
# Windows (Command Prompt)
set DB_PASSWORD=mysecret

# Windows (PowerShell)
$env:DB_PASSWORD = 'mysecret'

# Mac / Linux
export DB_PASSWORD=mysecret
```

Reading it in Python:

In [ ]:
import os

# os.environ is a dict-like object containing all environment variables
path_value = os.environ.get('PATH', 'not set')   # safe — returns default if not found
print(f'PATH is set: {bool(path_value)}')

# Read the DB password from the environment
db_password = os.environ.get('DB_PASSWORD', '')
if db_password:
    print('DB_PASSWORD is set (value hidden)')
else:
    print('DB_PASSWORD is not set — set it before running the pipeline')

---

## 4.6 .env Files

Setting environment variables in the terminal every time you open a new window is tedious. A **`.env` file** lets you list all your variables in one place, and a library called `python-dotenv` loads them automatically.

**Install:**
```
pip install python-dotenv
```

Create a file called `.env` in your project folder:
```
DB_HOST=localhost
DB_PORT=5432
DB_NAME=weather_db
DB_USER=postgres
DB_PASSWORD=mysecretpassword
```

**Critical rule:** Add `.env` to `.gitignore` so you never commit it. Your password must never appear in a Git repository.

In [ ]:
# Create a sample .env file for this lesson
# In real use you would fill in your actual password
dotenv_content = '''# Week 4 database credentials — do NOT commit this file
DB_HOST=localhost
DB_PORT=5432
DB_NAME=weather_db
DB_USER=postgres
DB_PASSWORD=your_password_here
'''

dotenv_path = Path('.env')
dotenv_path.write_text(dotenv_content, encoding='utf-8')
print('.env file created.')

In [ ]:
# Load the .env file using python-dotenv
try:
    from dotenv import load_dotenv
    load_dotenv()   # reads .env and sets the variables in os.environ
    print('Loaded .env file')
except ImportError:
    print('python-dotenv not installed. Run: pip install python-dotenv')

# Now os.environ.get() works for the values in .env
db_config = {
    'host':     os.environ.get('DB_HOST', 'localhost'),
    'port':     int(os.environ.get('DB_PORT', 5432)),
    'dbname':   os.environ.get('DB_NAME', 'weather_db'),
    'user':     os.environ.get('DB_USER', 'postgres'),
    'password': os.environ.get('DB_PASSWORD', ''),
}

print('DB config loaded (password hidden):')
for k, v in db_config.items():
    display = '***' if k == 'password' else v
    print(f'  {k}: {display}')

---

## 4.7 A Logged, Configured Pipeline

Putting it all together: a pipeline that reads its settings from `config.json`, reads secrets from `.env`, and logs every step.

In [ ]:
import time

def run_logged_pipeline(config_path='config.json'):
    '''Fetch weather for all cities, log every step, save to CSV.'''
    # --- Load config ---
    with open(config_path, 'r', encoding='utf-8') as f:
        cfg = json.load(f)

    # --- Set up logging ---
    log_dir = Path(cfg['log_dir'])
    log_dir.mkdir(exist_ok=True)
    logger, log_file = build_logger('weather_run', log_dir)
    logger.info('Pipeline started')
    logger.info(f'Config: {config_path}')

    # --- Load cities ---
    cities_file = Path(cfg['cities_file'])
    if not cities_file.exists():
        logger.error(f'Cities file not found: {cities_file}')
        return
    cities_df = pd.read_csv(cities_file)
    logger.info(f'Loaded {len(cities_df)} cities from {cities_file}')

    # --- Fetch weather ---
    api_cfg = cfg['api']
    all_frames = []

    for _, row in cities_df.iterrows():
        logger.info(f'Fetching {row["city"]}...')
        params = {
            'latitude':  row['latitude'],
            'longitude': row['longitude'],
            'daily':     api_cfg['fields'],
            'timezone':  row['timezone'],
        }
        try:
            response = requests.get(api_cfg['base_url'], params=params,
                                    timeout=api_cfg['timeout_sec'])
            response.raise_for_status()
            raw = response.json()

            df = pd.DataFrame(raw['daily']).rename(columns={
                'time':                'date',
                'temperature_2m_max':  'temp_max_c',
                'temperature_2m_min':  'temp_min_c',
                'precipitation_sum':   'rain_mm',
                'windspeed_10m_max':   'wind_max_kmh',
            })
            df['city']      = row['city']
            df['latitude']  = row['latitude']
            df['longitude'] = row['longitude']
            df['timezone']  = row['timezone']
            all_frames.append(df)
            logger.info(f'  Got {len(df)} rows for {row["city"]}')

        except requests.RequestException as e:
            logger.error(f'  FAILED for {row["city"]}: {e}')

        time.sleep(api_cfg['sleep_between_requests'])

    # --- Combine and save ---
    if not all_frames:
        logger.error('No data fetched — exiting')
        return

    weather = pd.concat(all_frames, ignore_index=True)
    weather['rain_mm'] = weather['rain_mm'].fillna(0.0)
    weather = weather.round(2)

    output_dir = Path(cfg['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / 'weather_forecast_logged.csv'
    weather.to_csv(out_path, index=False)

    logger.info(f'Saved {len(weather)} rows to {out_path}')
    logger.info('Pipeline finished successfully')
    return weather

result = run_logged_pipeline('config.json')
if result is not None:
    print(f'\nResult: {result.shape}')

In [ ]:
# Clean up files created in this lesson
for p in [config_path, dotenv_path]:
    if p.exists():
        p.unlink()
        print(f'Deleted {p.name}')

---

## Key Takeaways

1. **Use `logging` instead of `print()`** in any code that runs unattended. Log messages have timestamps and severity levels; `print()` has neither.
2. The five logging levels are: `DEBUG` → `INFO` → `WARNING` → `ERROR` → `CRITICAL`. Set the level to `INFO` in production to hide verbose debug output.
3. A **FileHandler** saves logs to a file. A **StreamHandler** writes to the console. Add both to one logger to get output in both places.
4. **Config files** (JSON) separate settings from code. Change a timeout or a URL by editing the file, not the code.
5. **Never put passwords or API keys in config files that go into Git.** Keep secrets in environment variables.
6. `os.environ.get('KEY', 'default')` reads an environment variable. Always supply a default so your code does not crash when the variable is missing.
7. A **`.env` file** + `python-dotenv` is the standard local development pattern. Add `.env` to `.gitignore` before your first commit.